In [3]:
import os
import pandas as pd
import usaddress

In [183]:
def parse_address(address):
    try:
        parsed = usaddress.tag(address)
        address_components = (parsed[0].get('AddressNumber', ''),
                              parsed[0].get('StreetName', ''),
                              parsed[0].get('StreetNamePostType', ''))
        return parsed[0]
    except usaddress.RepeatedLabelError as e:
        print(f"Error parsing address: {e}")
        return (None, None, None)

In [164]:
# Getting List of File Names in Links_By_County Directory
file_names = os.listdir("Links_By_County")
file_names

# Creating new DataFrame with all the data
df = pd.DataFrame()
for file in file_names:
    df = pd.concat([df, pd.read_excel(f"Links_By_County/{file}", header=0, index_col=0)])

df.sort_values(by=['County','Municipality'], inplace=True)
df['WIPPID'] = df['Link'].str.split('=').str[-1]

In [165]:
# Get List of files in downloaded_addys Directory
file_names = os.listdir("downloaded_addys")
file_names

# Creating new DataFrame with all the data
tdh_df = pd.DataFrame()
for file in file_names:
    county = file.split('.')[0]
    df_temp = pd.read_excel(f"downloaded_addys/{file}", header=0)
    # read excel while suppressing the warning    
    df_temp['County'] = county
    tdh_df = pd.concat([tdh_df, df_temp])

# Using usaddress to parse the address
#tdh_df['StructuredLocation'] = tdh_df['PropertyLocation'].apply(lambda x: parse_address(x))
#tdh_df['StructuredOwnerLocation'] = tdh_df['OwnerStreet'].apply(lambda x: parse_address(x))

/Users/sunilpc/Desktop/VSCode/LB_Screener/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/sunilpc/Desktop/VSCode/LB_Screener/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/sunilpc/Desktop/VSCode/LB_Screener/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/sunilpc/Desktop/VSCode/LB_Screener/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, app

In [166]:
# Get List of folders in Data_By_Towns Directory
folder_names = os.listdir("Data_By_Towns")

# Creating new DataFrame with all the data
wipp_df = pd.DataFrame()
for folder in folder_names:
    # Get List of files in each folder
    file_names = os.listdir(f"Data_By_Towns/{folder}")
    for file in file_names:
        # Read CSV then add column named "County" with the name of the folder
        df_temp = pd.read_csv(f"Data_By_Towns/{folder}/{file}")
        df_temp["County"] = folder
        # Append to the DataFrame
        wipp_df = pd.concat([wipp_df, df_temp])

# Using usaddress to parse the address
#wipp_df['StructuredLocation'] = wipp_df['Property Location'].apply(lambda x: parse_address(x))


In [167]:
'''
unique_keys_wipp     =     wipp_df['StructuredLocation'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()
unique_property_keys_tdh = tdh_df['StructuredLocation'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()
unique_owner_keys_tdh   =  tdh_df['StructuredOwnerLocation'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()

# Combine the unique keys
unique_keys = set(unique_keys_wipp).union(set(unique_property_keys_tdh), set(unique_owner_keys_tdh))
required_keys = ['AddressNumber', 'StreetName', 'StreetNamePostType']

for key in required_keys:
    wipp_df[f"Property{key}"] = wipp_df['StructuredLocation'].apply(lambda x: x.get(key, None) if x is not None else None)
    tdh_df[f"Property{key}"] = tdh_df['StructuredLocation'].apply(lambda x: x.get(key, None) if x is not None else None)
    tdh_df[f"Owner{key}"] = tdh_df['StructuredOwnerLocation'].apply(lambda x: x.get(key, None) if x is not None else None)

print(f"Unique keys: {unique_keys}")
'''

'\nunique_keys_wipp     =     wipp_df[\'StructuredLocation\'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()\nunique_property_keys_tdh = tdh_df[\'StructuredLocation\'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()\nunique_owner_keys_tdh   =  tdh_df[\'StructuredOwnerLocation\'].apply(lambda x: list(x.keys()) if x is not None else []).explode().unique()\n\n# Combine the unique keys\nunique_keys = set(unique_keys_wipp).union(set(unique_property_keys_tdh), set(unique_owner_keys_tdh))\nrequired_keys = [\'AddressNumber\', \'StreetName\', \'StreetNamePostType\']\n\nfor key in required_keys:\n    wipp_df[f"Property{key}"] = wipp_df[\'StructuredLocation\'].apply(lambda x: x.get(key, None) if x is not None else None)\n    tdh_df[f"Property{key}"] = tdh_df[\'StructuredLocation\'].apply(lambda x: x.get(key, None) if x is not None else None)\n    tdh_df[f"Owner{key}"] = tdh_df[\'StructuredOwnerLocation\'].apply(lambda x: x.get(key, None) if

In [168]:
replacement_codes = {
    'apt': 'unit',
    'drive': 'dr',
    'lane': 'ln',
    'avenue': 'ave',
    'boulevard': 'blvd',
    'street': 'st',
    'circle': 'cir',
    'road': 'rd',
    'court': 'ct',
    'parkway': 'pkwy',
    'square': 'sq',
    'trail': 'trl',
    'terrace': 'ter',
    'terr': 'ter',
    'highway': 'hwy',
    'expressway': 'expwy',
    'pike': 'pky',
    'way': 'wy',
    'alley': 'aly',
    'driveway': 'drvway',
    'building': 'bldg',
    'bridge': 'brg'
}


In [169]:
# Cleaning up tdh_df 

tdh_df['OwnerStreetStrip'] = tdh_df['OwnerStreet'].str.lower().replace(r'[^a-zA-Z0-9\s]', '', regex=True)
tdh_df['PropertyLocationStrip'] = tdh_df['PropertyLocation'].str.lower().replace(r'[^a-zA-Z0-9\s]', '', regex=True)

for key, value in replacement_codes.items():
    tdh_df['OwnerStreetStrip'] = tdh_df['OwnerStreetStrip'].str.replace(key,value)
    tdh_df['PropertyLocationStrip'] = tdh_df['PropertyLocationStrip'].str.replace(key,value)

tdh_df['Rental'] = tdh_df['OwnerStreetStrip'] != tdh_df['PropertyLocationStrip']
tdh_df = tdh_df[tdh_df['PropertyClassCode'] == '2']

tdh_df['PropertyLocationStrip'] = tdh_df['PropertyLocationStrip'].str.strip()

tdh_df

,PamsPin,OwnerName,OwnerStreet,OwnerCityState,OwnerZipCode,Block,Lot,Qual,PropertyLocation,PropertyClassCode,...,SR1A_Code,EPL_Code,Facility,InitialFilingDate,FurtherFilingDate,ExemptStatuteNumber,County,OwnerStreetStrip,PropertyLocationStrip,Rental
0,1801_27_7.08,"Shah, Arjun & Picarella, Mary E",16 Steeple Chase Court,Bedminster Nj,7921,27.0,7.08,NaN,16 Steeple Chase Court,2,...,7.0,0.0,NaN,01/01/0001,01/01/0001,NaN,Somerset,16 steeple chase ct,16 steeple chase ct,False
1,1801_59_1.76,"Vyas, Zeal M & Pratik N",12 Smoke Rise Ln,"Bedminster, Nj",7921,59.0,1.76,NaN,12 Smoke Rise Lane,2,...,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN,Somerset,12 smoke rise ln,12 smoke rise ln,False
2,1801_59_2.14,"Patel, Darshan",35 Revere Drive,"Bedminster, Nj",7921,59.0,2.14,NaN,35 Revere Drive,2,...,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN,Somerset,35 revere dr,35 revere dr,False
3,1801_59_2.17,"Govan, Dave & Laura J",29 Revere Drive,Bedminster Nj,7921,59.0,2.17,NaN,29 Revere Drive,2,...,1.0,0.0,NaN,01/01/0001,01/01/0001,NaN,Somerset,29 revere dr,29 revere dr,False
4,1801_59_2.25,"Desai, Darshit H & Richa D",15 Revere Dr,"Bedminster, Nj",7921,59.0,2.25,NaN,15 Revere Drive,2,...,0.0,0.0,NaN,01/01/0001,01/01/0001,NaN,Somerset,15 revere dr,15 revere dr,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,1616_111_13.14,Shah Shardh,880 Rifle Camp Rd,Woodland Park Nj,7424,111.0,13.14,NaN,880 Rifle Camp Rd,2,...,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN,Passaic,880 rifle camp rd,880 rifle camp rd,False
1490,1616_113_12.01_C0504,Coleen Rathod Residence Trust,120 Mezzine Dr,Cresskill Nj,7626,113.0,12.01,C0504,4 Slate Court B3,2,...,26.0,0.0,NaN,01/01/0001,01/01/0001,NaN,Passaic,120 mezzine dr,4 slate ct b3,True
1491,1616_114_2.05,Patel Dhaval & Jigisha,695 Rifle Camp Rd,Woodland Park Nj,7424,114.0,2.05,NaN,695 Rifle Camp Rd,2,...,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN,Passaic,695 rifle camp rd,695 rifle camp rd,False
1492,1616_124_6.01_C0208,Panchal Dipak & Reena,3 Coldstream Lane,Upper Saddle River Nj,7458,124.0,6.01,C0208,208 Woodland Drive 54,2,...,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN,Passaic,3 coldstream ln,208 woodland dr 54,True


In [170]:
# Cleaning up wipp_df 
wipp_df = wipp_df.drop(columns=['Unnamed: 0'])
wipp_df['PropertyLocationStrip'] = wipp_df['Property Location'].str.lower().replace(r'[^a-zA-Z0-9\s]', '', regex=True)

for key, value in replacement_codes.items():
    wipp_df['PropertyLocationStrip'] = wipp_df['PropertyLocationStrip'].str.replace(key,value)

wipp_df['PropertyLocationStrip'] = wipp_df['PropertyLocationStrip'].str.strip()

wipp_df

,Owner Name,Property Location,Municipality,Search,County,PropertyLocationStrip
0,"SHARMA, SUMITA & LAURE, JOSEPH",11 BROAD ST,Maurice River Township,Sharma,Cumberland,11 brd st
0,AMIN ANUPAM & MAIER REBECCA,226 LAMBERTVL HOPEWELL RD,Hopewell Township,Amin,Cumberland,226 lambertvl hopewell rd
1,AMIN ANUPAM & MAIER REBECCA,226 LAMBERTVL HOPEWELL RD,Hopewell Township,Amin,Cumberland,226 lambertvl hopewell rd
2,AMIN JATIN & PRANITI,27 DUBLIN RD,Hopewell Township,Amin,Cumberland,27 dublin rd
3,AMIN RAVI & NUPUR,2370 PENNINGTON RD,Hopewell Township,Amin,Cumberland,2370 pennington rd
...,...,...,...,...,...,...
357,"TRIVEDI, RAMAN & SEETA",317 TULIP COURT,Marlboro Township,Trivedi,Monmouth,317 tulip ct
358,"VAGHELA, PARANTAP & PARMAR, GUNJAN",4 DRAKES HILL COURT,Marlboro Township,Vaghela,Monmouth,4 drakes hill ct
359,"VAGHELA, SHRADDHA H",302 HARVARD PLACE,Marlboro Township,Vaghela,Monmouth,302 harvard place
360,"VYAS, DEEPA & PATHAK, JIGAR KUMAR",198 FRANKLIN PLACE,Marlboro Township,Vyas,Monmouth,198 franklin place


In [193]:
# Merging DataFrames on "PropertyLocationStrip"
merged_df = pd.merge(wipp_df, tdh_df, on=['PropertyLocationStrip', 'County'], how='outer')

# Dropping columns
dropping_columns = ['BuildingClass', 'BuildingDescription', 'LandDescription', 'Acreage', 'Additional Lots 1', 
                    'Additional Lots 2', 'MapPage', 'Zone', 'OldBlock', 'OldLot', 'OldQual', 'CurrentYearTaxes', 
                    'TaxCode1', 'TaxCode2', 'TaxCode3', 'TaxCode4', 'TaxAccountNumber', 'MortgageAccountNumber', 
                    'BankCode', 'DeedBook', 'DeedPage', 'PropertyUseCode', 'SR1A_Code', 'EPL_Code', 'Facility', 
                    'InitialFilingDate', 'FurtherFilingDate', 'ExemptStatuteNumber', 'LastYearTaxes', 'BuildingSqFt',
                    'Lot', 'Qual', 'SaleAssessment', 'Block']
merged_df = merged_df.drop(columns=dropping_columns)

# Sort by County
merged_df.sort_values(by=['PropertyLocationStrip'], inplace=True)

# Parsing address
merged_df['StructuredLocation'] = merged_df['PropertyLocationStrip'].apply(lambda x: parse_address(x))
merged_df = merged_df.drop_duplicates(subset=['County', 'PropertyLocationStrip'])

merged_df

,Owner Name,Property Location,Municipality,Search,County,PropertyLocationStrip,PamsPin,OwnerName,OwnerStreet,OwnerCityState,OwnerZipCode,PropertyLocation,PropertyClassCode,YearBuilt,SaleDate,SalePrice,OwnerStreetStrip,Rental,StructuredLocation
0,"SHAH, MINESH J & BHARTI M",0 VIRGINIA STREET,Sayreville Borough,Shah,Middlesex,0 virginia st,1219_198.01_16.03,"Shah, Minesh J & Bharti M",0 Virginia Street,"Sayreville, Nj",8872.0,0 Virginia Street,2,0.0,04/26/1996,212500.0,0 virginia st,False,"{'AddressNumber': '0', 'StreetName': 'virginia..."
1,"RANA, ILAXI",001 EAST SHORE RD,Morristown Town,Rana,Morris,001 east shore rd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'AddressNumber': '001', 'StreetNamePreDirecti..."
2,"JAIN, GAURAV/SONI, REEMA",003 NEWCASTLE CT,Morristown Town,Jain,Morris,003 newcastle ct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'AddressNumber': '003', 'StreetName': 'newcas..."
3,"SHAH, JAY P/SEJAL J",005 ROBERTS DR,Morristown Town,Shah,Morris,005 roberts dr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'AddressNumber': '005', 'StreetName': 'robert..."
4,"GANDHI, AMIT/NINA",006 ROBIN HOOD DR,Morristown Town,Gandhi,Morris,006 robin hood dr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'AddressNumber': '006', 'StreetName': 'robin ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32023,"SHARMA, PRAKESH C + YOGESH",WATSON LN,Fairfield Township,Sharma,Essex,watson ln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'StreetName': 'watson', 'StreetNamePostType':..."
32024,"PATEL, VINIT V ETAL",WEST SADDLE RIVER ROAD,Waldwick Borough,Patel,Bergen,west saddle river rd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'StreetNamePreDirectional': 'west', 'StreetNa..."
32025,"PATEL, SETURKUMAR & JINALBEN",WHITE HORSE PIKE,Galloway Township,Patel,Atlantic,white horse pky,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'AddressNumber': 'white', 'StreetName': 'hors..."
32026,"SHUKLA, BIPINCHANDRA R, ETAL",WILSON AVENUE,Hamilton Township,Shukla,Atlantic,wilson ave,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'StreetName': 'wilson', 'StreetNamePostType':..."


In [194]:
# Fill missing 'Owner Name' columns with 'OwnerName' column values
merged_df['Owner Name'] = merged_df['Owner Name'].fillna(merged_df['OwnerName'])
# Drop 'OwnerName' column
merged_df = merged_df.drop(columns=['OwnerName'])

# Fill missing 'Property Location' columns with 'OwnerStreet' column values
merged_df['Property Location'] = merged_df['Property Location'].fillna(merged_df['OwnerStreet'])
# Drop 'OwnerStreet' column
merged_df = merged_df.drop(columns=['OwnerStreet'])

# Fill missing 'Municipality' columns with 'OwnerCityState' column values
merged_df['Municipality'] = merged_df['Municipality'].fillna(merged_df['OwnerCityState'].str.split(',').str[0])
# Drop 'OwnerCityState' column
merged_df = merged_df.drop(columns=['OwnerCityState'])

# Drop PamsPin	OwnerZipCode	PropertyLocation	PropertyClassCode	YearBuilt	SaleDate	SalePrice	OwnerStreetStrip StructuredLocation columns
merged_df = merged_df.drop(columns=['PamsPin', 'OwnerZipCode', 'PropertyLocation', 'PropertyClassCode', 'YearBuilt', 'SaleDate', 'SalePrice', 'OwnerStreetStrip', 'StructuredLocation'])



,Owner Name,Property Location,Municipality,Search,County,PropertyLocationStrip,Rental
0,"SHAH, MINESH J & BHARTI M",0 VIRGINIA STREET,Sayreville Borough,Shah,Middlesex,0 virginia st,False
1,"RANA, ILAXI",001 EAST SHORE RD,Morristown Town,Rana,Morris,001 east shore rd,NaN
2,"JAIN, GAURAV/SONI, REEMA",003 NEWCASTLE CT,Morristown Town,Jain,Morris,003 newcastle ct,NaN
3,"SHAH, JAY P/SEJAL J",005 ROBERTS DR,Morristown Town,Shah,Morris,005 roberts dr,NaN
4,"GANDHI, AMIT/NINA",006 ROBIN HOOD DR,Morristown Town,Gandhi,Morris,006 robin hood dr,NaN
...,...,...,...,...,...,...,...
32023,"SHARMA, PRAKESH C + YOGESH",WATSON LN,Fairfield Township,Sharma,Essex,watson ln,NaN
32024,"PATEL, VINIT V ETAL",WEST SADDLE RIVER ROAD,Waldwick Borough,Patel,Bergen,west saddle river rd,NaN
32025,"PATEL, SETURKUMAR & JINALBEN",WHITE HORSE PIKE,Galloway Township,Patel,Atlantic,white horse pky,NaN
32026,"SHUKLA, BIPINCHANDRA R, ETAL",WILSON AVENUE,Hamilton Township,Shukla,Atlantic,wilson ave,NaN


In [195]:
# Sort by County, Municipality, PAM

merged_df.to_csv('merged_data.csv', index=False)

for county in merged_df['County'].unique():
    county_df = merged_df[merged_df['County'] == county]
    os.makedirs('County_Data', exist_ok=True)
    county_df.to_csv(f'County_Data/{county}_data.csv', index=False)

In [ ]:
# Get files in County_Data directory
county_files = os.listdir('County_Data')

api_call_count = 0

# Iterate through each file and get Longitude and Latitude
for file in county_files:
    print(file)
    df = pd.read_csv(f'County_Data/{file}')
    df['Latitude'] = None
    df['Longitude'] = None

    for index, row in df.iterrows():
        address = f'{row["Property Location"].lower()}, {row["Municipality"].lower()}, NJ'
        geocode = gmaps.geocode(address)
        api_call_count += 1
        if geocode:
            lat = geocode[0]['geometry']['location']['lat']
            lng = geocode[0]['geometry']['location']['lng']
            df.at[index, 'Latitude'] = lat
            df.at[index, 'Longitude'] = lng
        else:
            df.at[index, 'Latitude'] = None
            df.at[index, 'Longitude'] = None

In [3]:
string__ = '''
Cape May_data.csv
                         Owner Name Property Location          Municipality  \
0          DOSHI, NILESH & SANGEETA   10 77TH ST EAST         Sea Isle City   
1        PANDYA, MELIND R & PARUL M      10 HARRYS CT        Upper Township   
2                      Desai, Tapan  101 S 4Th St #A5            Rio Grande   
3  Espinosa, James A & Rupal Patel-  10 Oakwood Place              Voorhees   
4      SHAH, VARSHA & ARORA, MANALI   10726 THIRD AVE  Stone Harbor Borough   

   Search    County PropertyLocationStrip Rental   Latitude  Longitude  
0   Doshi  Cape May       10 77th st east   True  39.131002 -74.706512  
1  Pandya  Cape May          10 harrys ct  False  39.234696 -74.676091  
2     NaN  Cape May       101 s 4th st a5  False  39.010815 -74.872866  
3     NaN  Cape May        1011 wesley rd   True  39.819556 -74.915754  
4    Shah  Cape May       10726 third ave   True  39.046422 -74.766895  
API Calls: 93
Burlington_data.csv
                      Owner Name  Property Location           Municipality  \
0         SONI, MANISH & SHALINI   1 ARNCLIFFE RISE  Medford Lake Township   
1  PATEL, DHRUVINKUMAR AND NEHAL  1 ARROWHEAD DRIVE    Burlington Township   
2          Pal, Kanchan & Shukla       1 Biddle Way              Mt Laurel   
3   PATEL, RAJANKUMAR & PRAKRUTI    1 BRITTANY BLVD       Evesham Township   
4     PARIKH, NICOLE R & PARTH R   1 CAROLINE DRIVE  Medford Lake Township   

   Search      County PropertyLocationStrip Rental   Latitude  Longitude  
0    Soni  Burlington      1 arncliffe rise  False  39.829562  -74.81125  
1   Patel  Burlington        1 arrowhead dr  False  40.078957 -74.807204  
2     NaN  Burlington           1 biddle wy  False  39.940293 -74.921023  
3   Patel  Burlington       1 brittany blvd  False  39.911786 -74.882285  
4  Parikh  Burlington         1 caroline dr  False  39.903308   -74.8436  
API Calls: 1604
Salem_data.csv
                     Owner Name       Property Location         Municipality  \
0        MEHTA, VINOD + PURNIMA        1173 MALLARD WAY  Pittsgrove Township   
1        PATEL, ALPESH + SWETAL  123 RUNNING DEER TRAIL  Pittsgrove Township   
2  PARMAR, DHAVALKUMAR + LILLIE          126 HARVARD RD  Pennsville Township   
3   PATEL, AMRISH J + SANGITA A          145 N RIVER DR  Pennsville Township   
4                  PATEL PIYUSH          149 FORDHAM RD  Pennsville Township   

   Search County PropertyLocationStrip  Rental   Latitude  Longitude  
0   Mehta  Salem       1173 mallard wy     NaN   39.54396 -75.164826  
1   Patel  Salem  123 running deer trl     NaN  39.525991 -75.136223  
2  Parmar  Salem        126 harvard rd     NaN  39.644549 -75.531276  
3   Patel  Salem        145 n river dr     NaN  39.665252 -75.516251  
4   Patel  Salem        149 fordham rd     NaN  39.646492 -75.532597  
API Calls: 1631
Gloucester_data.csv
                            Owner Name Property Location         Municipality  \
0                PATEL, MITUL & VARSHA    1 ANNAMARIE CT    Woolwich Township   
1                DESAI, NIKHIL & RUPAL     1 BENJAMIN CT  Washington Township   
2   PATEL, HARDIKKUMAR & VADHER, USHMA      10 NICOLE CT    Deptford Township   
3              PATEL, KAMLESH & RESHMA     10 RAINBOW DR  Washington Township   
4  AMIN, BHUPENDRA & MEENABEN &HIMANSU    10 SKYHOOK CIR    Deptford Township   

  Search      County PropertyLocationStrip  Rental   Latitude  Longitude  
0  Patel  Gloucester        1 annamarie ct     NaN  39.716684 -75.356107  
1  Desai  Gloucester         1 benjamin ct     NaN  39.730639 -75.075737  
2  Patel  Gloucester          10 nicole ct     NaN  39.845104 -75.113903  
3  Patel  Gloucester         10 rainbow dr     NaN  39.720166 -75.072029  
4   Amin  Gloucester        10 skyhook cir     NaN  39.839167 -75.119956  
API Calls: 1952
Mercer_data.csv
                       Owner Name  Property Location           Municipality  \
0   VYAS, JALPABEN N & NIRAVKUMAR     1 ALLERTON WAY  East Windsor Township   
1    PATEL, TUSHAR S & RAJULBEN T  1 ARBORWOOD COURT  East Windsor Township   
2           MEHTA, PUNEET & SEEMA     1 ASHLEY COURT  East Windsor Township   
3                      SHAH AESHA     1 BEACON COURT           Trenton City   
4  PATEL JAYANTILAL A & SUSHILA J      1 BEARDSLY CT    Washington Township   

  Search  County PropertyLocationStrip  Rental   Latitude  Longitude  
0   Vyas  Mercer         1 allerton wy     NaN  40.246799 -74.535732  
1  Patel  Mercer        1 arborwood ct     NaN  40.244513 -74.536084  
2  Mehta  Mercer           1 ashley ct     NaN  40.252233 -74.527263  
3   Shah  Mercer           1 beacon ct     NaN  40.220179 -74.764229  
4  Patel  Mercer         1 beardsly ct     NaN  40.274291 -74.618169  
API Calls: 3645
Hunterdon_data.csv
                       Owner Name    Property Location         Municipality  \
0    Desai, Rachit M & Lam Nguyen         1 Amherst Ct         Annandale Nj   
1                  PATEL, BHISHMA     1 BALMORAL DRIVE  Alexandria Township   
2         PATEL, JAYANTI & NALINI  1 JUDGE THOMPSON RD  Readington Township   
3           SHAH, BHAVNA & MAYANK    1 LONGBOW TERRACE     Raritan Township   
4  BHATT, PARAG & ROMA ASHOK SHAH         1 MAGRIET RD  Readington Township   

  Search     County PropertyLocationStrip Rental   Latitude  Longitude  
0    NaN  Hunterdon          1 amherst ct  False  40.617323 -74.904021  
1  Patel  Hunterdon         1 balmoral dr  False  40.575768 -74.992054  
2  Patel  Hunterdon   1 judge thompson rd  False  40.577405 -74.718081  
3   Shah  Hunterdon         1 longbow ter  False  40.488134 -74.878388  
4  Bhatt  Hunterdon          1 magriet rd  False  40.623786 -74.771404  
API Calls: 3960
Morris_data.csv
                  Owner Name  Property Location     Municipality  Search  \
0                RANA, ILAXI  001 EAST SHORE RD  Morristown Town    Rana   
1   JAIN, GAURAV/SONI, REEMA   003 NEWCASTLE CT  Morristown Town    Jain   
2        SHAH, JAY P/SEJAL J     005 ROBERTS DR  Morristown Town    Shah   
3          GANDHI, AMIT/NINA  006 ROBIN HOOD DR  Morristown Town  Gandhi   
4  DESAI, SANJAY/IVETTE ABUD      019 BARTON RD  Morristown Town   Desai   

   County PropertyLocationStrip  Rental   Latitude  Longitude  
0  Morris     001 east shore rd     NaN  41.182554 -74.320593  
1  Morris      003 newcastle ct     NaN  40.877193 -74.434634  
2  Morris        005 roberts dr     NaN  40.893045 -74.455063  
3  Morris     006 robin hood dr     NaN    40.8798 -74.439197  
4  Morris         019 barton rd     NaN  40.947477  -73.86309  
API Calls: 4584
Hudson_data.csv
                    Owner Name      Property Location       Municipality  \
0             SHAH, ROSE MARIE  1 JACOB'S LANDING WAY  Secaucus Township   
1        PATEL,PADMESH & DIVYA        1 LIBERTY COURT  Secaucus Township   
2  SHAH, SANJAY & NIMISHA SHAH          1 LUHRS COURT  Secaucus Township   
3     PATEL, BHUPENDRA & HANSA             1 OAK LANE  Secaucus Township   
4          JOSHI MOHAN & MEGHA        10 CREEKSIDE CT  Secaucus Township   

  Search  County PropertyLocationStrip  Rental   Latitude  Longitude  
0   Shah  Hudson   1 jacobs landing wy     NaN  40.802467 -74.060077  
1  Patel  Hudson          1 liberty ct     NaN  40.794232 -74.059557  
2   Shah  Hudson            1 luhrs ct     NaN  40.801778 -74.057645  
3  Patel  Hudson              1 oak ln     NaN  40.804445 -74.057801  
4  Joshi  Hudson       10 creekside ct     NaN  40.800778 -74.048077  
API Calls: 5489
Sussex_data.csv
                        Owner Name        Property Location  \
0    JAIN, ANSHUL & WONG, JENNIFER  1 HILTON HEAD DR UNIT 8   
1                    PATEL, ARPITA          108 CONKLIN AVE   
2  PATHAK, HRISHI H & SHARMA, EKTA        111 PAHAQUARRY RD   
3       PATEL, JASHVANT A & RAJULA             118 METRO TR   
4       PATEL, JASHVANT A & RAJULA             122 METRO TR   

        Municipality  Search  County    PropertyLocationStrip  Rental  \
0    Vernon Township    Jain  Sussex  1 hilton head dr unit 8     NaN   
1  Hopatcong Borough   Patel  Sussex          108 conklin ave     NaN   
2  Hopatcong Borough  Pathak  Sussex        111 pahaquarry rd     NaN   
3  Hopatcong Borough   Patel  Sussex             118 metro tr     NaN   
4  Hopatcong Borough   Patel  Sussex             122 metro tr     NaN   

    Latitude  Longitude  
0  41.179803 -74.524518  
1  40.918954 -74.685023  
2  40.932931 -74.664744  
3  40.932348 -74.675498  
4  40.932592 -74.675144  
API Calls: 5528
Camden_data.csv
                     Owner Name      Property Location          Municipality  \
0  PATEL, TEJASH M & NIRMALABEN            1 BARBET DR     Voorhees Township   
1        BHATT JAIMIN & JAIMIKA  1 BIRCHWOOD PARK DR S  Cherry Hill Township   
2              KOTHARI, ASHOK S     1 BRITTON PL STE 6     Voorhees Township   
3    PATEL, ASHOKKUMAR B & TARA         1 COVINGTON LN     Voorhees Township   
4          PATEL BHUPENDRAKUMAR      1 FOX CHASE DRIVE   Gloucester Township   

    Search  County  PropertyLocationStrip Rental   Latitude  Longitude  
0    Patel  Camden            1 barbet dr   True  39.868257 -74.931632  
1    Bhatt  Camden  1 birchwood park dr s  False  39.904858 -74.958512  
2  Kothari  Camden     1 britton pl ste 6    NaN  39.844565 -75.003258  
3    Patel  Camden         1 covington ln  False  39.860554 -74.932203  
4    Patel  Camden         1 fox chase dr  False  39.784259 -75.019662  
API Calls: 6626
Ocean_data.csv
                  Owner Name      Property Location         Municipality  \
0  PATEL, AMIT & SHAH, AMITA            1 ALMA ROAD  Long Beach Township   
1            PARIKH, HEMISHA         1 CARRIAGE WAY    Barnegat Township   
2            TRIPATHI, MEERA          1 INVERELL DR     Berkely Township   
3  DESAI, HETAL I & TRIPTI H  1 KNIGHTSBRIDGE PLACE     Jackson Township   
4      PATEL, MITESH & HETAL           1 LOCH COURT       Lacey Township   

     Search County PropertyLocationStrip  Rental   Latitude  Longitude  
0     Patel  Ocean             1 alma rd     NaN  39.753718 -74.125095  
1    Parikh  Ocean         1 carriage wy     NaN   39.76549 -74.276817  
2  Tripathi  Ocean         1 inverell dr     NaN  39.975036 -74.280813  
3     Desai  Ocean    1 knightsbrg place     NaN  40.143204 -74.288461  
4     Patel  Ocean             1 loch ct     NaN  39.840056   -74.1913  
API Calls: 6910
Union_data.csv
                        Owner Name Property Location  \
0                    DESAI, HARDIK       1 GINESI DR   
1            PATEL, MANISH & JIGNA   1 LILLIAN COURT   
2    SHAH, HARIS ALI & NAZIA IQBAL      1 SAMOSET RD   
3  BHATT, GAURAV - CHAUHAN, SHIVEE  1 STATION SQUARE   
4                      PATEL, NEIL   10 & R NILES ST   

                Municipality Search County PropertyLocationStrip  Rental  \
0             Clark Township  Desai  Union           1 ginesi dr     NaN   
1  Berkeley Heights Township  Patel  Union          1 lillian ct     NaN   
2          Cranford Township   Shah  Union          1 samoset rd     NaN   
3             Union Township  Bhatt  Union          1 station sq     NaN   
4             Elizabeth City  Patel  Union        10  r niles st     NaN   

    Latitude  Longitude  
0  40.616951 -74.315226  
1  40.669992 -74.425097  
2  40.642588 -74.293268  
3  40.679159 -74.244152  
4  40.653215 -74.204426  
API Calls: 7515
Passaic_data.csv
                         Owner Name     Property Location     Municipality  \
0                      PATEL PRANAV    1 ALEXANDER AVENUE  Wanaque Borough   
1  LAKHANI ABDUL & ABBASI MAHNOOR Z         1 CEDAR COURT   Totowa Borough   
2                      GANDHI MAMTA   1 COMMANDER'S COURT   Totowa Borough   
3             MEHTA SAMIR D & NISHA  1 CONTINENTAL CIRCLE   Totowa Borough   
4            Rana, Mahesh & Jalpa M           1 Grant Ave      Clifton  Nj   

    Search   County PropertyLocationStrip Rental   Latitude  Longitude  
0    Patel  Passaic       1 alexander ave   True  41.031156 -74.303272  
1  Lakhani  Passaic            1 cedar ct  False   40.89565 -74.214606  
2   Gandhi  Passaic       1 commanders ct  False  40.902541 -74.223802  
3    Mehta  Passaic     1 continental cir  False  40.911694 -74.221864  
4      NaN  Passaic           1 grant ave  False   40.87975 -74.156491  
API Calls: 9003
Warren_data.csv
                     Owner Name   Property Location       Municipality  \
0      PATEL, MINESH & NITIKSHA        10 DALTON RD  Hackettstown Town   
1      TAILOR, TEJAL K & HINA T    10 ROBESON RIDGE    Oxford Township   
2  JAIN, ANKIT & POOJA BHANDARI   100 BERGEN STREET  Hackettstown Town   
3    PATEL, SURESH N & DIPITBEN  106 COUNTRYSIDE DR  Hackettstown Town   
4   TRIVEDI, BHARAT R & BIMAL B    11 MIDLAND DRIVE   Liberty Township   

    Search  County PropertyLocationStrip  Rental   Latitude  Longitude  
0    Patel  Warren          10 dalton rd     NaN  40.864314 -74.808166  
1   Tailor  Warren      10 robeson ridge     NaN  40.824603 -74.973809  
2     Jain  Warren         100 bergen st     NaN  40.852224  -74.83453  
3    Patel  Warren    106 countryside dr     NaN  40.864974 -74.831502  
4  Trivedi  Warren         11 midland dr     NaN  40.864811 -74.934461  
API Calls: 9068
Essex_data.csv
                        Owner Name Property Location          Municipality  \
0              SHAH, AJAY & PREETY      1 ALPINE WAY   Livingston Township   
1                   SHUKLA, NILESH     1 AMBROSIA CT   Livingston Township   
2            MEHTA, DINESH & ANILA     1 ARROW DRIVE   Livingston Township   
3            Essoka, Modi & Gloria    1 Arverne Road           West Orange   
4  SHAH SAMEET & BRAHMBHATT AASHKA     1 ASPEN DRIVE  Cedar Grove Township   

   Search County PropertyLocationStrip Rental   Latitude  Longitude  
0    Shah  Essex           1 alpine wy  False  40.772847 -74.304553  
1  Shukla  Essex         1 ambrosia ct  False  40.772802 -74.361711  
2   Mehta  Essex            1 arrow dr  False  40.774534 -74.333981  
3     NaN  Essex          1 arverne rd  False  40.777829 -74.275221  
4    Shah  Essex            1 aspen dr    NaN  40.853605 -74.241668  
API Calls: 10600
Bergen_data.csv
                       Owner Name     Property Location       Municipality  \
0        PATEL, KAUSHIK & BHAVANA    0-126 TUNBRIDGE RD  Fair Lawn Borough   
1  SHAH, AMISH V. & SHETH, BINITA       0-141 YERGER RD  Fair Lawn Borough   
2           PATEL, RAJESH K & AMI  0-21 WHITEHALL ST 1X  Fair Lawn Borough   
3      PATEL, JAIMIN H & TRUPTI J       0-28 30TH ST 1X  Fair Lawn Borough   
4  PATEL, ASHVIN K, JANAK & SUNIL          0-36 34TH ST  Fair Lawn Borough   

  Search  County PropertyLocationStrip  Rental   Latitude  Longitude  
0  Patel  Bergen        0126 tunbrg rd     NaN   40.92377 -74.097189  
1   Shah  Bergen        0141 yerger rd     NaN  40.923572 -74.098468  
2  Patel  Bergen   021 whitehall st 1x     NaN  40.918783 -74.112275  
3  Patel  Bergen        028 30th st 1x     NaN  40.920697 -74.104715  
4  Patel  Bergen           036 34th st     NaN  40.922232  -74.10108  
API Calls: 11699
Monmouth_data.csv
                          Owner Name   Property Location        Municipality  \
0               PATEL, MANOJ & MAMTA   1 BALTUSROL DRIVE  Manalapan Township   
1            PATEL, AMRISH & URSHILA       1 BORDEAUX LN    Holmdel Township   
2  PATHAK, NIKITA & KALPESH D. PATEL         1 BOYD ROAD     Hazlet Township   
3              PATEL, CHETAN & HEENA  1 BUCKINGHAM DRIVE  Manalapan Township   
4         TRIVEDI, PRABHAS & NIRMALA   1 CAVALCADE COURT  Manalapan Township   

    Search    County PropertyLocationStrip  Rental   Latitude  Longitude  
0    Patel  Monmouth        1 baltusrol dr     NaN  40.261416 -74.373281  
1    Patel  Monmouth         1 bordeaux ln     NaN  40.348578 -74.163689  
2   Pathak  Monmouth             1 boyd rd     NaN  40.413964 -74.188683  
3    Patel  Monmouth       1 buckingham dr     NaN  40.268147 -74.346658  
4  Trivedi  Monmouth        1 cavalcade ct     NaN  40.264111 -74.354866  
API Calls: 12847
Somerset_data.csv
                    Owner Name Property Location          Municipality  \
0  PATHAK, SHIV KUMAR & POONAM    1 ALLEGHENY DR     Bernards Township   
1             AMIN, PUSHPAVATI     1 ARTHUR ROAD  Bridgewater Township   
2         DOSHI, RAJESH & YATI     1 AVERY COURT  Bridgewater Township   
3                 GANDHI, MINA     1 BARMOUTH CT     Franklin Township   
4              PATEL, HARISH B      1 BERING WAY     Franklin Township   

   Search    County PropertyLocationStrip Rental   Latitude  Longitude  
0  Pathak  Somerset        1 allegheny dr  False   40.64774 -74.612456  
1    Amin  Somerset           1 arthur rd  False  40.616452 -74.625884  
2   Doshi  Somerset            1 avery ct  False  40.590825 -74.673062  
3  Gandhi  Somerset         1 barmouth ct  False  40.519047 -74.505895  
4   Patel  Somerset           1 bering wy  False  40.485805 -74.560974  
API Calls: 16544
Cumberland_data.csv
                        Owner Name      Property Location       Municipality  \
0        SHARMA PRAKASH C & YOGESH         1 RUSTIC DRIVE  Hopewell Township   
1           DAVE RAJ & BHALA ANITA  10 LAKE BALDWIN DRIVE  Hopewell Township   
2             JOSHI SHIWANI & AMIT          100 DARROW DR  Hopewell Township   
3  SHAH HARISHCHANDRA & REKHABEN H          101 GENTRY CT  Hopewell Township   
4            PATHAK KEDAR & KETAKI       101 HADDON COURT  Hopewell Township   

   Search      County PropertyLocationStrip  Rental   Latitude  Longitude  
0  Sharma  Cumberland           1 rustic dr     NaN  40.344547 -74.747286  
1    Dave  Cumberland    10 lake baldwin dr     NaN  40.342554 -74.783331  
2   Joshi  Cumberland         100 darrow dr     NaN  40.359178 -74.755808  
3    Shah  Cumberland         101 gentry ct     NaN  40.295952 -74.772947  
4  Pathak  Cumberland         101 haddon ct     NaN  40.304884 -74.774346  
API Calls: 16724
Atlantic_data.csv
'''

# print lines ending with .csv in the string
for line in string__.split('\n'):
    if line.endswith('.csv'):
        print(line)

Cape May_data.csv
Burlington_data.csv
Salem_data.csv
Gloucester_data.csv
Mercer_data.csv
Hunterdon_data.csv
Morris_data.csv
Hudson_data.csv
Sussex_data.csv
Camden_data.csv
Ocean_data.csv
Union_data.csv
Passaic_data.csv
Warren_data.csv
Essex_data.csv
Bergen_data.csv
Monmouth_data.csv
Somerset_data.csv
Cumberland_data.csv
Atlantic_data.csv
